# 26 · Калибровка P(событие ≤ 24ч), isotonic per-bin

Цель: сделать вероятности честными (P=0.4 значит 40% реальных событий).
Протокол без утечки: калибратор строится на **валидации** (2025H1), применяется
к **holdout** (≥ 2025-07-01). Также: из-за редких событий isotonic строим по
децилям `p24` (per-bin) — только на валидационных точках.

Для каждой задачи (fire/access/sensor):
1. обучаем/загружаем модель, считаем `p24` на val → подгоняем IsotonicRegression;
2. применяем к holdout `p24`;
3. сравниваем O/E по децилям и ECE до/после;
4. сохраняем калибраторы в `models/calib_<task>.pkl`.


In [1]:
%matplotlib inline
import sys
import pathlib
import pickle

_HERE = pathlib.Path.cwd()
RESEARCH = _HERE.parent if _HERE.name == "notebooks" else _HERE
if str(RESEARCH) not in sys.path:
    sys.path.insert(0, str(RESEARCH))

import numpy as np
import pandas as pd

from catboost import CatBoostClassifier
from sklearn.isotonic import IsotonicRegression

import data_utils as du
import tte_pipeline as tte


DATASET = du.find_dataset_dir()
MODELS = DATASET.parent / "models"

def load_model(task):
    m = CatBoostClassifier(task_type="CPU")
    m.load_model(MODELS / f"tte_{task}_discrete_hazard.cbm")
    return m


In [2]:
def val_risk30(task, model):
    s = pd.read_parquet(DATASET / f"tte_subjects_{task}.parquet")
    _, va, _ = tte.split_subjects(s)
    Xv, _ = tte.expand_person_time(va, tte.subject_features(va),
                                   tte.HORIZON_BUCKETS, fixed_horizon=True)
    p = model.predict_proba(Xv)[:, 1]
    S = np.cumprod(1 - p.reshape(len(va), tte.HORIZON_BUCKETS), axis=1)
    r30 = (1 - S[:, -1]).astype(np.float64)
    y30 = ((va["event_flag"] == 1) & (va["obs_days"] <= 30.01)).to_numpy(int)
    return r30, y30

def fit_bin_map(p, y, bins=20, min_events=100):
    # per-bin калибровка на риск30 (много событий) — стабильно
    q = np.quantile(p, np.linspace(0, 1, bins + 1))
    q[0] -= 1e-9
    q[-1] += 1e-9
    bp = np.clip(np.digitize(p, q) - 1, 0, bins - 1)
    xq, rates = [], []
    for b in range(bins):
        m = bp == b
        n = int(m.sum())
        if n == 0 or int(y[m].sum()) < max(1, min_events):
            continue
        xq.append(p[m].min())
        rates.append(float(y[m].mean()))
    xq.append(np.inf)
    return (np.asarray(xq), np.asarray(rates))

def apply_bin_map(cal, p):
    xq, rates = cal
    idx = np.asarray(np.searchsorted(xq, p, side="right") - 1)
    return np.clip(rates[idx.astype(int)], 0.0, 1.0)

TA = {
    "fire": "Пожарный риск",
    "access": "Несанкц. доступ",
    "sensor": "Отказ датчика",
}
CAL30 = {}
for task, name in TA.items():
    print(f"=== {name} ({task}) риск30 ===", flush=True)
    model = load_model(task)
    r30v, y30v = val_risk30(task, model)
    cal = fit_bin_map(r30v, y30v)
    CAL30[task] = cal
    print("калибратор risk30: событий val:", int(y30v.sum()),
          "| бинов:", len(cal[1]), flush=True)
    with open(MODELS / f"calib30_{task}.pkl", "wb") as f:
        pickle.dump(cal, f)
print("калибраторы risk30 сохранены: models/calib30_*.pkl")


=== Пожарный риск (fire) риск30 ===


калибратор risk30: событий val: 5761 | бинов: 5


=== Несанкц. доступ (access) риск30 ===


калибратор risk30: событий val: 14862 | бинов: 20


=== Отказ датчика (sensor) риск30 ===


калибратор risk30: событий val: 4324 | бинов: 14


калибраторы risk30 сохранены: models/calib30_*.pkl


In [3]:
def oe_table(p, y, bins=10):
    idx = np.argsort(p)
    n = len(p)
    rows = []
    for i in range(bins):
        sl = idx[i * n // bins:(i + 1) * n // bins]
        pred = p[sl].mean()
        obs = y[sl].mean()
        rows.append({"pred": pred, "obs": obs,
                     "O/E": obs / pred if pred > 0 else np.nan, "n": len(sl)})
    return pd.DataFrame(rows)

def ece(p, y, bins=10):
    idx = np.argsort(p)
    n = len(p); acc = 0.0
    for i in range(bins):
        sl = idx[i * n // bins:(i + 1) * n // bins]
        acc += np.abs(p[sl].mean() - y[sl].mean()) * len(sl) / n
    return acc

SAMPLE = {}
report = {}
for task, name in TA.items():
    ho = pd.read_parquet(MODELS / f"tte_{task}_holdout.parquet")
    y30h = ((ho["event_flag"] == 1) & (ho["obs_days"] <= 30.01)).to_numpy(int)
    r30h = ho["risk30"].to_numpy(np.float64)
    r30c = apply_bin_map(CAL30[task], r30h)
    before = oe_table(r30h, y30h)
    after = oe_table(r30c, y30h)
    print(f"=== {name} ({task}) риск30 ===")
    print("ECEs: до {:.4f} -> после {:.4f}".format(
        ece(r30h, y30h), ece(r30c, y30h)))
    print("средн. |O/E-1|: до {:.3f} -> после {:.3f}".format(
        np.nanmean(np.abs(before["O/E"] - 1)), np.nanmean(np.abs(after["O/E"] - 1))))
    print("до:"); print(before[["pred", "obs", "O/E"]].round(4).to_string(index=False))
    print("после:"); print(after[["pred", "obs", "O/E"]].round(4).to_string(index=False))
    print()
    report[task] = {"ece_before": ece(r30h, y30h), "ece_after": ece(r30c, y30h)}
pd.DataFrame(report).T.to_csv(DATASET / "calibration30_summary.csv",
                              encoding="utf-8-sig")
print("summary30 -> dataset/calibration30_summary.csv")


=== Пожарный риск (fire) риск30 ===
ECEs: до 0.0781 -> после 0.3260
средн. |O/E-1|: до 0.668 -> после 2.450
до:
  pred    obs    O/E
0.0052 0.0000 0.0000
0.0074 0.0000 0.0000
0.0102 0.0000 0.0000
0.0234 0.0035 0.1494
0.0562 0.0430 0.7649
0.1285 0.1630 1.2686
0.2219 0.6280 2.8306
0.2947 0.2840 0.9637
0.3965 0.3250 0.8197
0.7136 0.5115 0.7168
после:
  pred    obs    O/E
0.0504 0.3867 7.6783
0.0504 0.3910 7.7638
0.0622 0.2935 4.7208
0.0707 0.2660 3.7630
0.3363 0.2165 0.6439
0.4680 0.0490 0.1047
0.4680 0.0425 0.0908
0.4680 0.0485 0.1036
0.4680 0.0475 0.1015
0.5687 0.2170 0.3816

=== Несанкц. доступ (access) риск30 ===
ECEs: до 0.0212 -> после 0.0157
средн. |O/E-1|: до 0.219 -> после 0.151
до:
  pred    obs    O/E
0.0263 0.0400 1.5198
0.0395 0.0440 1.1148
0.0495 0.0560 1.1319
0.0623 0.1055 1.6936
0.0849 0.1180 1.3907
0.1247 0.1350 1.0823
0.1776 0.1675 0.9433
0.2377 0.2190 0.9212
0.3637 0.3845 1.0571
0.8570 0.8055 0.9399
после:
  pred    obs    O/E
0.0467 0.0515 1.1037
0.0536 0.0555 1.0353
0

### Как это встраивать
- В API инференса: `p24_raw` из модели → `calib.predict(p24_raw)` — единая точка
  калибровки (файл `models/calib_<task>.pkl`), потом пороги/топ-K по честным
  вероятностям (ожидаемые объёмы диспетчеру считаются через `p̂`).
- Ранжирование (Uno-C / PR-AUC) не меняется — калибровка меняет только
  абсолютный масштаб вероятностей.
